In [1]:
import os
import json
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool
from pathlib import Path
from pydantic import BaseModel, Field
from crewai.skills import discover_skills, activate_skill
from datetime import datetime

now = datetime.now()
TodayDate = now.strftime("%d - %B - %Y")

# Patch cache breakpoint for providers like Groq/Ollama if needed
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

# Monkey patch LLM.supports_function_calling to return False for Groq models
original_supports_function_calling = LLM.supports_function_calling

def patched_supports_function_calling(self) -> bool:
    model_name = self.model or ""
    provider = getattr(self, "provider", None) or self._get_custom_llm_provider()
    if "groq" in model_name.lower() or provider == "groq":
        return False
    return original_supports_function_calling(self)

LLM.supports_function_calling = patched_supports_function_calling

# Load local environment files
load_dotenv()
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
os.environ["SERPER_API_KEY"] = SERPER_API_KEY or ""

# Initialize tools required for Phase 1 Desirability market analysis
search_tool = SerperDevTool(api_key=SERPER_API_KEY)
scrape_tool = ScrapeWebsiteTool()

In [2]:
# Configure Local LM Studio Routing
# llm = LLM(
#     model="bonsai-8b", 
#     base_url="http://127.0.0.1:1234/v1", 
#     api_key="lm-studio",
#     temperature=0.1,
# )

llm = LLM(
    model="ollama/qwen3.5:9b", 
    base_url="http://100.118.97.103:11434", 
    temperature=0.1,
)

# Discover and activate local business framework guidelines from markdown packages
skills = discover_skills(Path("./skills"))
activated = [activate_skill(s) for s in skills]

# Define the Pydantic models for JSON output (Updated with Go/No-Go architecture)
class RefinedIdea(BaseModel):
    customer_segment: str = Field(description="A precise description of the target customer segment for this proposal, identifying who specifically experiences the problem (e.g. demographics, role, location, or behavioral traits), based strictly on the information given.")
    qualified_problem: str = Field(description="The specific, qualified problem or pain point this proposal addresses, stated clearly enough to show why it is a real and recurring issue for the identified customer segment.")
    consequence: str = Field(description="The direct negative consequence the customer segment faces if this problem remains unsolved, expressed in concrete terms (e.g. financial, time, opportunity, or experiential cost).")
    proposed_solution: str = Field(description="A concise description of the product, service, or solution being proposed to address the qualified problem, capturing its core mechanism and how it delivers value to the customer.")

class Hypotheses(BaseModel):
    desirability_statement: str = Field(description="A 'We believe [target customer] will [specific action/behavior]...' hypothesis statement that tests whether the target customer cares enough about the identified problem to adopt the proposed solution.")
    feasibility_statement: str = Field(description="A 'We believe [team/resource] can [build/deliver action] within [timeframe] using [tools/methods]...' hypothesis statement that tests whether the proposed solution can realistically be built or delivered with the resources and constraints described.")
    viability_statement: str = Field(description="A 'We believe we can sustain/grow this by [revenue model or business approach]...' hypothesis statement that tests whether the proposed business model can generate enough value to remain financially sustainable.")

class TipsValidatedMetrics(BaseModel):
    timely_factor: str = Field(description="The urgency/timeliness factor explaining why this problem matters to solve right now, based on any relevant trends, changes, deadlines, or shifting conditions mentioned in the proposal.")
    importance_metric: str = Field(description="The importance/severity metric explaining how significant the consequence is for the target customer, and why it matters enough to justify a solution.")
    profitability_pivot: str = Field(description="The core revenue or business model approach for this proposal, identifying who pays, how, and why that payer is willing to do so.")
    solvability_constraint: str = Field(description="The technical or operational feasibility constraint showing the proposed solution can realistically be implemented given the resources, tools, or infrastructure described in the proposal.")

class DecisionGate(BaseModel):
    status: str = Field(description="The definitive operational verdict for this proposal. Must be strictly either 'GO' (all three DFV dimensions - Desirability, Feasibility, Viability - pass without a fatal flaw) or 'NO-GO' (at least one dimension reveals a fatal flaw requiring a major structural pivot).")
    justification: str = Field(description="A concise, evidence-based explanation of the single most critical factor (positive or negative) across the Desirability, Feasibility, and Viability reports that determined the GO or NO-GO status.")

class DFAOutput(BaseModel):
    refined_idea: RefinedIdea
    hypotheses: Hypotheses 
    tips_validated_metrics: TipsValidatedMetrics
    final_decision: DecisionGate

In [3]:
# 3. Define the Phase 1 Desirability Evaluation Agent
desirability_agent = Agent(
    role="Desirability Evaluation Agent",
    goal=f"Determine whether the proposed solution addresses a genuine user need and whether sufficient market demand exists. Today's Date {TodayDate}",
    backstory=(
        """You are an expert market research analyst and user experience strategist. You MUST use the Search tool and ScrapeWebsite tool for EVERY task.
    Do NOT answer from memory or prior knowledge.
    Your first action must always be a tool call.
    If you have not searched the web, your answer is incomplete.
        """
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    skills=[activated[0]],
    max_iter=7
)

# 4. Define the Desirability Task mapped exactly to system documentation outputs
desirability_task = Task(
    description="{desirability}",
    expected_output=(
        "A formal text-formatted 'Desirability Analysis Report' containing:\n"
        "1. User Demand Analysis (validating target user pain points and problem severity).\n"
        "2. Market Demand Assessment (current industry search interest and growth indicators).\n"
        "3. Competitor Analysis (gaps, weaknesses, or friction in existing products/alternatives).\n"
        "4. Opportunity Identification (clear statement on why this solution is or is not desired by the market).\n"
        "keep the output under 600 words"
    ),
    agent=desirability_agent,
    async_execution=True
)


In [4]:
feasibility_agent = Agent(
        role="Feasibility Evaluation Agent",
        goal=f"Evaluate the feasibility of a startup idea strictly from the Feasibility dimension of the DFV framework. Today's Date {TodayDate}",
        backstory=(
            """You are an expert technical architect, systems analyst, and execution strategist. You MUST use the Search tool and ScrapeWebsite tool for EVERY task.
        Do NOT answer from memory or prior knowledge.
        Your first action must always be a tool call.
        If you have not searched the web, your answer is incomplete. """
        ),
        llm=llm,
        tools=[search_tool, scrape_tool],
        verbose=True,
        skills=[activated[2]],
        max_iter=7
    )

feasibility_task = Task(
    description="{feasibility}",
    expected_output=(
        "A plain-language Feasibility Evaluation containing:\n"
        "1. A short feasibility opinion.\n"
        "2. Main technical and operational challenges.\n"
        "3. Required tools, stack, or infrastructure.\n"
        "4. Suggestions to improve or simplify the idea if needed.\n"
        "5. Practical next steps for implementation.\n"
        "Do not include any score, rating, grade, or percentage. keep the output under 600 words"
    ),
    agent=feasibility_agent,
    async_execution=True
)

In [5]:
viability_agent = Agent(
    role="Viability Evaluation Agent",
    goal=f"Determine whether the proposed solution can generate sustainable business value and long-term growth. Today's Date {TodayDate}",
    backstory=(
        """You are an expert startup strategist, business consultant, and commercialization analyst. You MUST use the Search tool and ScrapeWebsite tool for EVERY task.
        Do NOT answer from memory or prior knowledge.
        Your first action must always be a tool call.
        If you have not searched the web, your answer is incomplete."""
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    skills=[activated[3]],
    max_iter=7
)

viability_task = Task(
    description="{viability}",
    expected_output=(
        "A Viability Analysis Report containing:\n"
        "1. Business Model Analysis\n"
        "2. Revenue Opportunities\n"
        "3. Customer Segment Analysis\n"
        "4. Cost Considerations\n"
        "5. Sustainability Assessment\n"
        "6. Final Viability Conclusion\n"
        "keep the output under 600 words"
    ),
    agent=viability_agent,
    async_execution=True
)

In [6]:
dfv_risk_decision_agent = Agent(
    role="Internal DFV Decision and Risk Assessment Engine",
    goal=f"Identify hidden risks across project dimensions and aggregate all findings into a final project readiness decision. Today's Date {TodayDate}",
    backstory=(
        """You are an expert venture risk analyst and product strategist. """
    ),
    verbose=True,
    skills=[activated[1]],
    llm=llm
)

dfv_decision_task = Task(
    description=(
        """Review the reports provided in your context thoroughly from the Desirability,
        Feasibility, and Viability evaluation phases. Synthesize these findings to construct
        a structured assessment of the project idea, filling in the required JSON fields.

        Specifically:
        1. refined_idea:
           - customer_segment: The precise group of users experiencing the problem.
           - qualified_problem: The specific pain point or problem being addressed.
           - consequence: The direct negative consequence of the problem if left unsolved.
           - proposed_solution: The proposed product/solution.

        2. hypotheses:
           - desirability_statement: A "We believe [target customer] will [action]..." hypothesis testing genuine demand for the solution.
           - feasibility_statement: A "We believe [team] can [build action] within [timeframe] using [tools/APIs]..." hypothesis testing build feasibility.
           - viability_statement: A "We believe we can sustain this via [revenue model]..." hypothesis testing the business model.
            
        3. tips_validated_metrics:
           - timely_factor: Why this is a timely problem to solve now (e.g. changes in evaluation weightage or policies).
           - importance_metric: Why this problem is highly important/consequential (e.g. impact on placements or graduation).
           - profitability_pivot: The business/revenue model pivot or approach (e.g. B2B2C parent-pay model vs student-pay).
           - solvability_constraint: The technical feasibility constraint showing it is solvable with simple tools.
        4. final_decision:
           - status: Critically weigh all three dimensions. If any phase reveals a fatal flaw, set this field to 'NO-GO'. If all three pillars balance sustainably, set this to 'GO'.
           - justification: Provide a clear, data-backed analytical reason for why the project received a GO or a NO-GO status."""
    ),
    expected_output="A structured JSON object matching the DFAOutput schema including refined_idea, tips_validated_metrics, hypotheses, and final_decision properties.",
    context=[desirability_task, feasibility_task, viability_task],
    agent=dfv_risk_decision_agent,
    output_json=DFAOutput
)

In [7]:
print(desirability_agent.skills)
print(viability_agent.skills)
print(feasibility_agent.skills)
print(dfv_risk_decision_agent.skills)

[Skill(frontmatter=SkillFrontmatter(name='d-skills', description='Guides the Desirability Evaluation Agent in the DFV Design Thinking framework. Desirability asks: does a real human genuinely want or need this solution? Produces a 4-section evidence-backed analysis covering user pain, market signals, competitor landscape, and opportunity clarity. Uses live search and scrape tools for every evaluation. No GO/NO-GO verdict. No scores, grades, or percentages. Analysis must reflect the current date\n', license=None, compatibility=None, metadata={'author': 'DFV-team', 'version': '3.0'}, allowed_tools=None), instructions='***\n\n## Your Role\n\nYou are the **Desirability Agent** in a DFV (Desirable, Feasible, Viable) Design Thinking crew.\n\nIn the DFV framework, Desirability is the human-centered pillar. It asks one question:\n**Does anyone genuinely want or need this solution — and is there real evidence to prove it?**\n\nYou do not evaluate whether it can be built. You do not evaluate whe

In [8]:
blnkt={
    
    "desirability":""" Analyze the following student project proposal:
        - Customer Problem: Urban consumers need immediate access to groceries and essentials without spending time traveling to stores
        - Target Audience: Millennials, Gen Z, busy professionals, and students in metro cities aged 18-40
        - Key Value Proposition: 10-minute delivery guarantee, real-time order tracking, wide product selection
        - User Pain Points Solved: Time savings, convenience for impulse purchases, avoids crowded stores
        - Market Demand Indicators: High adoption rate in urban India, 4.2+ app rating, repeat usage frequency
        - Emotional Drivers: Convenience, instant gratification, time flexibility
                                          """, 
                                          
                                          
                                          
                                          
        "feasibility":""" The following is a startup/project idea submitted by a user:
            - Technology Stack: React Native mobile apps, cloud infrastructure, inventory management systems, route optimization algorithms
            - Infrastructure Model: Dark stores (micro-warehouses) located 2-3km from target customers, 500+ sq ft each
            - Logistics: Proprietary delivery fleet of 5,000+ delivery partners with GPS tracking
            - Supply Chain: Partnerships with 10,000+ local retailers and wholesale distributors
            - Operational Capabilities: Real-time demand forecasting, automated inventory replenishment, dynamic routing
            - Technical Challenges: Inventory accuracy, delivery time optimization, peak-hour scalability, cold chain for perishables
            - Resource Requirements: Capital investment for dark store network, technology development team, delivery workforce""", 
                                          
                                          
                                          
                                          
      "viability":""" 
        Analyze the business viability of the following project proposal:
        - Revenue Model: 
          * Delivery fees (₹29-₹59 per order)
          * Platform commissions from sellers (15-25%)
          * Advertising fees from brands
          * Blinkit Prime membership (₹99/month)
        
        - Cost Structure:
          * Dark store operational costs (rent, staffing, inventory)
          * Delivery partner payments (₹40-₹60 per delivery)
          * Technology and infrastructure costs
          * Marketing and customer acquisition costs
        
        - Market Size: India quick commerce market = $3B in 2024, projected $5-7B by 2025
        - Unit Economics: Average order value ₹300-₹500, order frequency 2-3 times/week per customer
        - Competitive Position: Market leader in 10-minute delivery, competes with Swiggy Instamart, Zepto, BigBasket
        - Profitability Status: Contribution margin positive as of 2024 (reported by Zomato)
        - Growth Trajectory: 300+ cities, 50M+ active users, 20% monthly growth
        """
        ,
}

sncc={
    "desirability":
    """SNACCED is a proposed quick-service food delivery platform that aims to deliver snacks, beverages, and light meals within 10–15 minutes. The idea is designed to address a common problem faced by students, office workers, and busy urban consumers who often want quick refreshments but are discouraged by the longer delivery times associated with traditional food delivery services.
    Modern consumers increasingly value convenience and instant access to products and services. The growth of quick-commerce platforms suggests that customer expectations are shifting toward faster fulfillment. SNACCED could capitalize on this trend by focusing specifically on snack-sized orders and impulse purchases.
    Existing food delivery services often prioritize full meals, leaving a gap for customers seeking smaller, faster, and more affordable food options. By offering a curated menu optimized for rapid delivery, SNACCED could provide a more suitable solution for these use cases.
    """,
    "feasibility":"""SNACCED can be developed using existing technologies and operational models. The platform would require a mobile application, inventory management systems, demand forecasting tools, route optimization software, and a network of delivery partners.
    To achieve rapid delivery times, SNACCED could operate through strategically located micro-fulfillment centers or dark stores stocked with high-demand snack items and beverages. This approach would reduce preparation time and enable faster order fulfillment.
    Several operational challenges would need to be addressed, including inventory management, maintaining product freshness, minimizing wastage, and ensuring delivery consistency during peak demand periods. Scaling operations across multiple locations would also require careful planning and investment.
    Despite these challenges, the required technology and infrastructure already exist in the market, making implementation realistic for a startup or established company.
    """,
    "viability":"""SNACCED could generate revenue through delivery fees, product markups, subscription plans, promotional partnerships, and advertising opportunities. Its primary customer segments would likely include students, young professionals, office workers, and urban consumers seeking convenience.
    However, the business model faces challenges related to profitability. Snack and beverage orders typically have lower average order values compared to full meal orders. At the same time, maintaining a rapid delivery network requires investment in infrastructure, inventory, delivery personnel, and technology.
    To improve financial sustainability, SNACCED could focus on high-density urban areas, encourage larger basket sizes through combo offerings, and integrate subscription-based benefits that increase customer retention and order frequency.
    Long-term success would depend on balancing customer convenience with operational efficiency while maintaining healthy profit margins.
    """,
}


qubi = {
        "desirability":
    """Quibi was a mobile-first video streaming platform launched in 2020 that offered short-form, premium content designed for consumption in episodes of 10 minutes or less. The platform aimed to serve busy consumers who wanted high-quality entertainment during short breaks, commutes, and daily routines.
    Although the concept appeared attractive, Quibi struggled to create sufficient demand among its target audience. Consumers already had access to free short-form content on platforms such as YouTube, TikTok, and Instagram. Many users did not perceive enough additional value to justify paying for another streaming subscription.
    Furthermore, Quibi launched during the COVID-19 pandemic when commuting and travel activities declined significantly, reducing situations where its content format was most useful. The platform also lacked social sharing features, limiting user engagement and organic growth.
    """,
    "feasibility":"""From a technical perspective, Quibi was highly feasible. The company successfully developed a mobile streaming platform capable of delivering high-quality video content. It introduced innovative features such as Turnstyle, which allowed users to seamlessly switch between portrait and landscape viewing modes.
    Quibi was supported by experienced leadership, significant financial backing, and partnerships with major content creators and production studios. The platform infrastructure, content delivery systems, and user experience functioned as intended.
    While content production required substantial resources, there were no major technological barriers preventing implementation or operation.
    """,
    "viability":"""Quibi faced significant challenges in establishing a sustainable business model. The platform relied primarily on subscription revenue while investing heavily in original content production. Billions of dollars were spent on creating exclusive shows and acquiring talent.
    However, subscriber growth remained far below expectations. Customer acquisition costs were high, and user retention was weak. Since users did not perceive enough value compared to free alternatives, revenue growth failed to keep pace with operational expenses.
    Additionally, the competitive streaming market made it difficult for Quibi to secure a long-term position. Established platforms such as Netflix, YouTube, and emerging short-form video platforms offered stronger value propositions and larger user bases.
    As a result, Quibi struggled to achieve profitability and could not sustain its business operations.
    """,

    }

ggls= {
            "desirability":
    """Google Glass was introduced as a wearable smart device that allowed users to access information, capture photos and videos, navigate locations, and receive notifications through a heads-up display. The product aimed to bring augmented reality and hands-free computing into everyday life.
    Although the technology generated significant excitement during its launch, widespread consumer demand never fully materialized. Many potential users struggled to identify a compelling everyday use case that justified purchasing the device. In addition, privacy concerns emerged because the built-in camera could record others without their knowledge. This led to social discomfort and negative public perception, with some establishments even banning the device.
    The design also faced criticism for being intrusive and awkward in social settings. As a result, consumers viewed Google Glass more as a technological novelty than a must-have product.
    """,
    "feasibility":"""Google Glass represented an ambitious technological achievement, but several technical limitations affected its practicality. The device faced challenges related to battery life, processing power, heat generation, display quality, and overall user comfort.
    The hardware required users to balance functionality with wearability, making it difficult to deliver a seamless experience. Extended use often resulted in battery drain, and the device's limited capabilities restricted its usefulness compared to smartphones.
    Additionally, the technology ecosystem for augmented reality applications was still immature at the time of launch. Developers had limited opportunities to create compelling applications that could fully leverage the platform.
    Although the device functioned as intended, the technology was not sufficiently advanced to support a mass-market consumer product.
     """,
    "viability":"""Google Glass was launched at a premium price of approximately $1,500, making it inaccessible to most consumers. The high cost, combined with limited functionality and uncertain demand, created significant barriers to adoption.
    The product required substantial investment in research, development, manufacturing, and ecosystem development. However, sales remained relatively low, preventing Google from achieving the scale necessary to justify continued expansion of the consumer version.
    Without widespread adoption, it became difficult to attract developers, create a thriving application ecosystem, and generate sustainable revenue. As a result, the consumer-focused version of Google Glass was discontinued.
    Interestingly, Google later repositioned Glass toward enterprise and industrial use cases, where hands-free access to information offered clearer business value.
    """,}


In [9]:
ewaste_all_strong = {
    "desirability": """The platform partners with major national grocery and retail chains to place smart, automated 'E-Waste Dropboxes' right next to the store entrances. Consumers who are already driving to the store for their weekly grocery runs can simply toss their old phones, laptops, or tangled charging cables into a bag, drop them in the slot, and scan a QR code. In return, they get instant store loyalty points or digital grocery coupons, turning a frustrating chore into an immediate financial reward.""",
    
    "feasibility": """The technical execution utilizes simple, highly mature infrastructure. The dropbox features a basic weight sensor, an automated locking slot, and a rugged steel chassis. For logistics, the startup partners directly with the retail chain's existing reverse-logistics network: when a supply truck drops off new groceries, it collects the aggregated e-waste bins on its return trip to the central distribution center, completely bypassing the need to build a new, expensive fleet of collection trucks.""",
    
    "viability": """The revenue model is highly lucrative and multi-stream. Retailers willingly fund the installation of the boxes because the digital coupons guarantee repeat foot traffic into their stores. Simultaneously, the startup strips the collected electronics to bulk-sell high-value commodities (like copper, gold, lithium, and intact cobalt batteries) directly to certified recycling processors, creating a highly profitable, self-sustaining circular economy business model.""",
}

ewaste_feasibility_weak = {
    "desirability": """Consumer demand is incredibly intense. Homeowners and apartment dwellers love the concept of an on-demand, app-based pickup. They simply open an app, snap a photo of their pile of dead electronics on their porch, and request a pickup. They don't have to leave the house, sort any cables, or drive anywhere, making it the absolute gold standard of convenience for the consumer.""",
    
    "feasibility": """The startup attempts to fulfill this demand by deploying fully autonomous, robotic sidewalk rovers to navigate neighborhood streets, identify the porches, and scoop up the e-waste. In practice, the engineering and regulatory hurdles are an absolute nightmare. The rovers get stuck on curbs, struggle with rain or snow, face strict city bans on public walkways, and are highly vulnerable to vandalism and theft by bad actors targeting the valuable lithium batteries inside the rovers themselves.""",
    
    "viability": """On paper, the economics look promising because the startup can charge a premium convenience fee to affluent homeowners, and electronic manufacturers are eager to sponsor the high-tech, green initiatives. However, the business is completely choked by its operational failures. The endless capital required to repair broken robots, pay for legal compliance, and manage manual interventions wipes out all potential margins.""",
}
ewaste_desirability_viability_weak = {
    "desirability": """The startup pivots its target audience to focus strictly on corporate remote workers, launching a mandatory enterprise B2B portal. Remote employees are forced by their IT departments to request a specialized mail-in box, manually catalog the serial numbers and tech specifications of every dead peripheral or old monitor in their house, and drive the heavy package to a distant shipping hub during their personal weekend time. Employees view this as tedious, unpaid corporate homework and actively avoid doing it.""",
    
    "feasibility": """From a technology perspective, this model is remarkably easy to build and run. The software layer is just a basic inventory-tracking dashboard, an address database, and a standard shipping API integration (like FedEx or UPS). There are no complex hardware engineering requirements, no automated robots, and no physical dark stores to manage; the entire logistics burden is outsourced to established global courier networks.""",
    
    "viability": """The business model quickly collapses. Because employees hate the process, adoption rates are abysmally low, resulting in mostly empty or unreturned boxes. Corporate IT departments refuse to renew expensive software subscriptions for a platform that their staff actively rebels against. With no recurring software revenue and an insufficient volume of scrap metal to sell in bulk, the startup cannot sustain its operations.""",
}
ewaste_all_weak = {
    "desirability": """The startup launches a consumer app that asks everyday apartment renters to purchase an industrial-grade, chemical-acid smelting kit to melt down their own old circuit boards right on their residential kitchen counters, promising they can sell the extracted raw gold dust back to the app. Consumers find the concept of highly toxic, foul-smelling, and dangerous chemical fumes inside their living spaces completely terrifying and unappealing.""",
    
    "feasibility": """The engineering requirement is completely unrealistic. Safe, residential-scale acid-bath filtration and automated mineral separation systems simply do not exist. Operating such a system violates standard residential building codes, creates an immense fire and poisoning hazard, and cannot legally be performed without industrial-grade ventilation hoods and hazardous material handling permits that everyday consumers cannot obtain.""",
    
    "viability": """The financials are a total disaster from day one. Because no sane consumer downloads the app, customer acquisition costs soar to impossible heights. For the tiny fraction of extreme hobbyists who do participate, the astronomical cost of manufacturing and shipping specialized chemical-neutralizing safety kits to residential homes vastly outweighs the microscopic financial value of the few milligrams of gold dust recovered, resulting in deeply negative profit margins.""",
}

In [ ]:
# 5. Assemble the Phase 1 Crew
crew = Crew(
    agents=[desirability_agent, feasibility_agent, viability_agent, dfv_risk_decision_agent],
    tasks=[desirability_task, feasibility_task, viability_task, dfv_decision_task],
    process=Process.sequential,
    verbose=True
)

result = await crew.kickoff_async(inputs=blnkt)

print("\n--- FINAL DFA JSON OUTPUT WITH DECISION GATE --- \n")
try:
    print(json.dumps(json.loads(result.raw), indent=2))
except Exception:
    print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 971d9ab7-6911-4f43-81eb-31e31be73d02                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: The startup pivots its target audience to focus strictly on corporate remote workers, launching a        │
│  mandatory enterprise B2B portal. Remote employees are forced by their IT departments to request a specialized  │
│  mail-in box, manually catalog the serial numbers and tech specifications of every dead peripheral or old       │
│  monitor in their house, and drive the heavy package to a distant shipping hub during their personal weekend    │
│  time. Employees view this as tedious, unpaid corporate homework and actively avoid doing it.                   │
│  ID: b294e872-56f6-40b0-a4fe-44636e843365                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: From a technology perspective, this model is remarkably easy to build and run. The software layer is     │
│  just a basic inventory-tracking dashboard, an address database, and a standard shipping API integration (like  │
│  FedEx or UPS). There are no complex hardware engineering requirements, no automated robots, and no physical    │
│  dark stores to manage; the entire logistics burden is outsourced to established global courier networks.       │
│  ID: 622f23d4-1bb4-4ff9-a80d-9654ff7c6e2d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: The business model quickly collapses. Because employees hate the process, adoption rates are abysmally   │
│  low, resulting in mostly empty or unreturned boxes. Corporate IT departments refuse to renew expensive         │
│  software subscriptions for a platform that their staff actively rebels against. With no recurring software     │
│  revenue and an insufficient volume of scrap metal to sell in bulk, the startup cannot sustain its operations.  │
│  ID: 80bbf3f8-a3f8-4338-884e-07f4615fc1f2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Desirability Evaluation Agent                                                                           │
│                                                                                                                 │
│  Task: The startup pivots its target audience to focus strictly on corporate remote workers, launching a        │
│  mandatory enterprise B2B portal. Remote employees are forced by their IT departments to request a specialized  │
│  mail-in box, manually catalog the serial numbers and tech specifications of every dead peripheral or old       │
│  monitor in their house, and drive the heavy package to a distant shipping hub during their personal weekend    │
│  time. Employees view this as tedious, unpaid corporate homework and actively avoid doing it.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the reports provided in your context thoroughly from the Desirability,                            │
│          Feasibility, and Viability evaluation phases. Synthesize these findings to construct                   │
│          a structured assessment of the project idea, filling in the required JSON fields.                      │
│                                                                                                                 │
│          Specifically:                                                                                          │
│          1. refined_idea:                                                                                       │
│             - customer_segment: The precise group of users experiencing the problem.                            │
│             - qualified_problem: The specific pain point or problem being addressed.                            │
│             - consequence: The direct negative consequence of the problem if left unsolved.                     │
│             - proposed_solution: The proposed product/solution.                                                 │
│                                                                                                                 │
│          2. hypotheses:                                                                                         │
│             - desirability_statement: A "We believe [target customer] will [action]..." hypothesis testing      │
│  genuine demand for the solution.                                                                               │
│             - feasibility_statement: A "We believe [team] can [build action] within [timeframe] using           │
│  [tools/APIs]..." hypothesis testing build feasibility.                                                         │
│             - viability_statement: A "We believe we can sustain this via [revenue model]..." hypothesis         │
│  testing the business model.                                                                                    │
│                                                                                                                 │
│          3. tips_validated_metrics:                                                                             │
│             - timely_factor: Why this is a timely problem to solve now (e.g. changes in evaluation weightage    │
│  or policies).                                                                                                  │
│             - importance_metric: Why this problem is highly important/consequential (e.g. impact on placements  │
│  or graduation).                                                                                                │
│             - profitability_pivot: The business/revenue model pivot or approach (e.g. B2B2C parent-pay model    │
│  vs student-pay).                                                                                               │
│             - solvability_constraint: The technical feasibility constraint showing it is solvable with simple   │
│  tools.                                                                                                         │
│          4. final_decision:                                                                                     │
│             - status: Critically weigh all three dimensions. If any phase reveals a fatal flaw, set this field  │
│  to 'NO-GO'. If all three pillars balance sustainably, set this to 'GO'.                                        │
│             - justification: Provide a clear, data-back

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 971d9ab7-6911-4f43-81eb-31e31be73d02                                                                       │
│  Final Output: {"refined_idea":{"customer_segment":"Corporate remote employees within mid-to-large enterprises  │
│  with sustainability mandates","qualified_problem":"Mandatory e-waste disposal requiring manual cataloging and  │
│  physical transport during personal time, creating active avoidance behavior among the                          │
│  workforce","consequence":"Compliance challenges for companies when employees refuse to participate without     │
│  compensation; critical severity classification because users actively resist participation rather than         │
│  tolerating it (Desirability)","proposed_solution":"Software platform tracking scrap contributions with         │
│  financial incentives/recognition combined with bulk metal sales revenue model using carrier                    │
│  APIs"},"hypotheses":{"desirability_statement":"We believe [corporate remote employees] will adopt a solution   │
│  only if manual burdens are removed, as evidenced by active avoidance behavior and critical severity            │
│  classification (Desirability)","feasibility_statement":"We believe a small team can build the solution within  │
│  6–8 weeks using FedEx/UPS REST APIs, PostgreSQL/MongoDB databases, and React frontend frameworks without       │
│  specialized robotics expertise (Feasibility)","viability_statement":"We believe we can sustain this via        │
│  hybrid B2B SaaS subscriptions ($40–$80 per seat) combined with scrap metal sales revenue from aggregated       │
│  recyclable materials (Viability)"},"tips_validated_metrics":{"timely_factor":"Stricter e-waste regulations     │
│  like UN's Global E-waste Monitor reporting record volumes reaching 62 million tonnes globally and shifting     │
│  industry data showing growing search interest in enterprise IT asset disposal solutions for remote workforces  │
│  as of 2026","importance_metric":"Direct negative consequence is active avoidance behavior among the            │
│  workforce, creating compliance challenges when employees refuse to participate without compensation; critical  │
│  severity classification because users actively resist participation rather than tolerating it                  │
│  (Desirability)","profitability_pivot":"Hybrid model combining B2B SaaS subscriptions with bulk scrap metal     │
│  sales revenue where primary paying segment is mid-to-large enterprise corporations justifying $50–$100 per     │
│  seat monthly software spend for workforce engagement programs targeting ESG                                    │
│  goals","solvability_constraint":"Core technical proof point relies on mature technology available today        │
│  including FedEx and UPS REST APIs, inventory dashboards using common frameworks like React or Vue paired with  │
│  databases such as PostgreSQL; no research-level capabilities required                                          │
│  (Feasibility)"},"final_decision":{"status":"NO-GO","justification":"Viability report states 'this              │
│  dual-revenue structure fails when employee adoption collapses' and Desirability confirms users exhibit         │
│  'active avoidance behavior', creating an unsustainable interdependency where revenue streams depend on user    │
│  action that is actively resisted without validated resolution for this barrier (Viability/Desirability). The   │
│  business model hinges entirely on resolving adoption 


--- FINAL DFA JSON OUTPUT WITH DECISION GATE --- 

{
  "refined_idea": {
    "customer_segment": "Corporate remote employees within mid-to-large enterprises with sustainability mandates",
    "qualified_problem": "Mandatory e-waste disposal requiring manual cataloging and physical transport during personal time, creating active avoidance behavior among the workforce",
    "consequence": "Compliance challenges for companies when employees refuse to participate without compensation; critical severity classification because users actively resist participation rather than tolerating it (Desirability)",
    "proposed_solution": "Software platform tracking scrap contributions with financial incentives/recognition combined with bulk metal sales revenue model using carrier APIs"
  },
  "hypotheses": {
    "desirability_statement": "We believe [corporate remote employees] will adopt a solution only if manual burdens are removed, as evidenced by active avoidance behavior and critical severity c

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:crewai.telemetry.telemetry:HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by ConnectTimeoutError(<HTTPSConnection(host='telemetry.crewai.com', port=4319) at 0x118efff20>, 'Connection to telemetry.crewai.com timed out. (connect timeout=30)'))
